In [4]:
from sentence_transformers import SentenceTransformer

import numpy as np

c:\proyecto\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
documentos =  [
    "El cliente reporta que su tarjeta de débito fue rechazada en un cajero automático",
    "Solicitud de aumento de línea de crédito para cuenta empresarial",
    "El cliente no reconoce un cargo de $150 en su estado de cuenta",
    "Consulta sobre requisitos para abrir una cuenta de ahorros infantil",
    "Reclamo por doble cobro en transferencia interbancaria",
    "Solicitud de copia de estado de cuenta de los últimos 6 meses",
    "El cajero automático retuvo la tarjeta del cliente",
    "Consulta sobre tasa de interés para préstamo hipotecario",
    "Cliente reporta fraude: transacciones no reconocidas en su tarjeta de crédito",
    "Solicitud para cambiar el número de teléfono asociado a la cuenta",
    "El cliente reporta una transacción que no ha realizado con su tarjeta de crédito"
    "No está funcionando las transferencias ACH a las 08:00 am",
    "El usuario de banca en línea fue bloqueado y no puede entrar"
    "La app móvil se encuentra fuera de servicio",
    "El cliente solicita un préstam por consolidación de deudas",
    "La empresa ABC quiere cancelar su cuenta de cheques y abrir una nueva",
    "Se han incrementado las solicitudes de tarjeta de crédto para clientes con historial crediticio limitado",
    "Consulta para cancelar una tarjeta de crédito que ha sido robada",
    "El usuario de la app no puede ver el saldo de la tarjeta de crédito",
    "Solicitud para préstamo de pequeña empresa para capital de trabajo"
]

In [6]:
modelo = SentenceTransformer('all-MiniLM-L6-v2')

embedding_docs = modelo.encode(documentos)

print(embedding_docs)  # (20, 384)

c:\proyecto\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\amyma\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\proyecto\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-s

[[-0.06877156  0.10031479 -0.08481806 ...  0.04159868  0.04261937
  -0.06902242]
 [-0.07836507 -0.0021639  -0.05692661 ... -0.02384484  0.01243766
  -0.00693656]
 [ 0.01442391  0.06243165 -0.03300641 ...  0.01453752 -0.02530422
  -0.05378464]
 ...
 [-0.06144952  0.09286618 -0.06421605 ... -0.00931071 -0.03279879
  -0.07937694]
 [-0.05287359  0.05848562 -0.03923473 ...  0.03621318  0.00560703
  -0.02533059]
 [ 0.01582055  0.02215908  0.01121994 ... -0.00916468  0.02633378
  -0.001639  ]]


In [7]:
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
def buscar_documentos_similares(consulta, documentos, embedding_docs, modelo, top_k=3):

    # 1. Convierte la pregunta a embedding
    embedding_consulta = modelo.encode([consulta])

    # 2. Calcula la similitud entre todos los documentos y la consulta
    similitudes = cosine_similarity(embedding_consulta, embedding_docs)[0]

    # 3. Obtén los índices de los documentos más similares
    indices_top = np.argsort(similitudes)[::-1][:top_k]

    # 4. Devuelve los documentos más similares y sus puntuaciones de similitud
    resultados = [(documentos[i], similitudes[i]) for i in indices_top]

    return resultados

In [9]:
pregunta = "alguien uso mi tarjeta sin permiso"

resultados = buscar_documentos_similares(pregunta, documentos, embedding_docs, modelo, top_k=3)

for texto, score in resultados:
    print(f"Documento: {texto}\nSimilitud: {score:.4f}\n")

Documento: El cajero automático retuvo la tarjeta del cliente
Similitud: 0.5455

Documento: El usuario de la app no puede ver el saldo de la tarjeta de crédito
Similitud: 0.4449

Documento: El cliente no reconoce un cargo de $150 en su estado de cuenta
Similitud: 0.4370



In [10]:
import time
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

client = QdrantClient(host="localhost", port=6333)
COL = "tickets_banco"

# Crear colección (la recrea si ya existía)
if client.collection_exists(COL):
    client.delete_collection(COL)

client.create_collection(
    COL, vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

c:\proyecto\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:287: UserWarning: Qdrant client version 1.20.0 is incompatible with server version 1.12.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


True

In [11]:
t0 = time.perf_counter()
client.upsert(COL, points=[
    PointStruct(id=i, vector=emb.tolist(), payload={"texto": doc})
    for i, (doc, emb) in enumerate(zip(documentos, embedding_docs))
])
print(f"Carga: {time.perf_counter() - t0:.3f} s")
print("Puntos en Qdrant:", client.count(COL).count)

Carga: 0.058 s
Puntos en Qdrant: 18


In [12]:
def buscar_qdrant(consulta, top_k=3):
    vector = modelo.encode(consulta).tolist()
    res = client.query_points(COL, query=vector, limit=top_k)
    return [(p.payload["texto"], p.score) for p in res.points]

for texto, score in buscar_qdrant(pregunta):
    print(f"Documento: {texto}\nSimilitud: {score:.4f}\n")

Documento: El cajero automático retuvo la tarjeta del cliente
Similitud: 0.5455

Documento: El usuario de la app no puede ver el saldo de la tarjeta de crédito
Similitud: 0.4449

Documento: El cliente no reconoce un cargo de $150 en su estado de cuenta
Similitud: 0.4370



In [13]:
preguntas_prueba = [
    ("alguien uso mi tarjeta sin permiso", "Cliente reporta fraude: transacciones no reconocidas en su tarjeta de crédito"),
    ("me quedé sin poder entrar a la banca en línea", "El usuario de banca en línea fue bloqueado y no puede entrar"),
    ("quiero un crédito para mi negocio", "Solicitud para préstamo de pequeña empresa para capital de trabajo"),
    ("el cajero se tragó mi tarjeta", "El cajero automático retuvo la tarjeta del cliente"),
    ("necesito mis estados de cuenta anteriores", "Solicitud de copia de estado de cuenta de los últimos 6 meses"),
]

In [14]:
def evaluar(buscar_fn, preguntas, top_k=3):
    aciertos = 0
    tiempos = []
    for pregunta, esperado in preguntas:
        t0 = time.perf_counter()
        resultados = buscar_fn(pregunta, top_k)
        tiempos.append(time.perf_counter() - t0)
        textos = [r[0] for r in resultados]
        if esperado in textos:
            aciertos += 1
    print(f"Aciertos en top-{top_k}: {aciertos}/{len(preguntas)}")
    print(f"Latencia promedio: {sum(tiempos)/len(tiempos)*1000:.1f} ms")

evaluar(buscar_qdrant, preguntas_prueba)

Aciertos en top-3: 1/5
Latencia promedio: 46.2 ms
